# Phase 3 — Multimodal RAG System & 3-Year Corporate EV Strategy (end-to-end)

**Task 3.1 (Pipeline Engineering)** — an operational multimodal RAG over one shared cross-modal CLIP-ViT-B/32 space (512-d), ingesting:

| Asset type | Docs | Embedding |
|---|---|---|
| **Text assets** — Phase 1 final text-analysis corpus (`ev_bus_text_analysis_final.csv`, 548 rows) | 548 | CLIP text |
| **Image corpus** — Phase 2 `image_rag_documents.jsonl` (39 images: caption + OCR + features) | 39 | CLIP text ⊕ CLIP image, fused |
| **Statista charts** — real Statista PNG exports (`statista/`) | 6 | CLIP image ⊕ caption, fused (**visual layout** indexed) |
| **IEA market trends** — sourced facts (IEA GEVO 2024/25, BNEF, MoHI) + rendered chart layouts | 10 + 6 | CLIP text / fused |

**Task 3.2 (Executive Query Testing)** — 8 multi-layered strategic queries, retrieval + `gpt-4o-mini` synthesis grounded strictly in retrieved evidence.

**Strategic Output** — Comprehensive 3-Year Corporate EV Strategy across the four board pillars (market positioning, infrastructure partnerships, battery supply-chain risk, digital marketing spend).

> Heavy logic lives in `scripts/rag_lib.py`. Video assets are intentionally out of scope for this run.


In [1]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
import rag_lib as R

OUT_Q = ROOT / "outputs" / "executive_queries"; OUT_Q.mkdir(parents=True, exist_ok=True)
OUT_S = ROOT / "outputs" / "strategy";          OUT_S.mkdir(parents=True, exist_ok=True)
print("Phase 3 root:", ROOT)

Phase 3 root: /home/u_admin/ev_bus/Phase_3_RAG_and_Strategy


## 1 · Task 3.1 — Build the cross-modal index

One vector per document, all in the same CLIP space, so a text query retrieves images, charts, market facts, and comments jointly. Image and chart docs fuse their visual embedding with their textual description (mean, re-normalised) — this is what indexes the *visual layout*, not just the caption.

In [2]:
mat, meta = R.build_index(verbose=True)

/home/u_admin/ev_bus/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 6730.04it/s]

  image docs:      39 (text⊕image fused)


  chart layouts:   12 (image⊕caption fused)


  market text:     10 (IEA/BNEF/MoHI)


  text corpus:     548 (from ev_bus_text_analysis_final.csv)
Index: 609 vectors x 512-d  {'image': 39, 'chart': 12, 'market_text': 10, 'text': 548}


### Retrieval sanity check — cross-modal hits for a text query

In [3]:
for q in ["India electric bus penetration rate forecast chart",
          "battery supply chain risk and China dependence",
          "consumer complaints about EV bus reliability and service"]:
    print("Q:", q)
    for s, m in R.retrieve(q, k=5):
        print(f"   {s:.3f}  {m['modality']:12s} {str(m['doc_id'])[:48]}")
    print()

Q: India electric bus penetration rate forecast chart
   0.821  market_text  mkt_iea_india_salesshare
   0.819  market_text  mkt_iea_india_stock
   0.800  text         c_f9f6547d4eb13e256195
   0.799  market_text  mkt_iea_india_fund
   0.781  text         c_6baea38a95f738ddcb39

Q: battery supply chain risk and China dependence
   0.885  text         c_62f5af4a3b0926899117
   0.884  text         c_8168e51104973da0d5d2
   0.867  text         c_4a3bafd14dec6a9e5d11
   0.867  market_text  mkt_supply_chain_risk
   0.862  text         c_7de9f3bce9bb03809f60

Q: consumer complaints about EV bus reliability and service
   0.874  text         c_6a9a7042354b204892d5
   0.873  text         c_b0d91afb531e8af190b1
   0.863  text         c_2f8a437b9a7ab575be08
   0.863  text         c_529bc0280cec5b65e64b
   0.860  text         c_0ff761b629b7ad250d21



## 2 · Task 3.2 — Executive query testing

8 multi-layered strategic queries. Retrieval = cosine top-K over the shared index; synthesis = `gpt-4o-mini` (temperature 0.2) under a strict grounding system prompt (no fabrication; market data vs narrative-framing sentiment kept distinct; small-sample flags).

In [4]:
results = {}
md_lines = ["# Phase 3 · Task 3.2 — Executive Query Testing (Multimodal RAG)\n",
            f"**Index:** {mat.shape[0]} docs x {mat.shape[1]}-d shared CLIP space. ",
            f"**Synthesiser:** `{R.LLM}` grounded in retrieved evidence.\n\n---\n"]
for qid, q in R.EXEC_QUERIES:
    hits, ans = R.answer_query(q, k=6)
    mods = {}
    for s, m in hits: mods[m["modality"]] = mods.get(m["modality"], 0) + 1
    print(f"{qid}: top1={hits[0][0]:.3f} modalities={mods}")
    results[qid] = {"query": q,
                    "retrievals": [{"sim": round(s,3), "doc_id": m["doc_id"], "modality": m["modality"]} for s, m in hits],
                    "llm_answer": ans}
    md_lines += [f"## {qid} — {q}\n",
                 "**Top-6 retrievals:** " + ", ".join(f"`{str(m['doc_id'])[:20]}`({m['modality'][:3]},{s:.2f})" for s, m in hits) + "\n",
                 ans + "\n\n---\n"]
(OUT_Q / "executive_queries.md").write_text("\n".join(md_lines))
(OUT_Q / "executive_queries.json").write_text(json.dumps(results, indent=2))
print("\nWrote", OUT_Q / "executive_queries.md")

Q1: top1=0.893 modalities={'market_text': 3, 'text': 3, 'image': 2, 'chart': 1}


Q2: top1=0.921 modalities={'market_text': 3, 'text': 3, 'chart': 1, 'image': 2}


Q3: top1=0.853 modalities={'text': 5, 'market_text': 1, 'image': 2, 'chart': 1}


Q4: top1=0.876 modalities={'market_text': 1, 'text': 5, 'chart': 1, 'image': 2}


Q5: top1=0.884 modalities={'text': 5, 'market_text': 1, 'chart': 1, 'image': 2}


Q6: top1=0.900 modalities={'text': 6, 'market_text': 1, 'image': 2, 'chart': 1}


Q7: top1=0.876 modalities={'text': 6, 'market_text': 1, 'image': 2, 'chart': 1}


Q8: top1=0.867 modalities={'text': 6, 'market_text': 1, 'image': 2, 'chart': 1}

Wrote /home/u_admin/ev_bus/Phase_3_RAG_and_Strategy/outputs/executive_queries/executive_queries.md


### Example answer (Q4 — battery supply chain)

In [5]:
print(results["Q4"]["llm_answer"])

1. **Direct answer**: The Indian OEM faces significant battery supply-chain risks due to heavy dependence on China for cell manufacturing and LFP cathode supply. While falling battery prices improve tender economics, they do not mitigate the risks associated with reliance on a single country for critical components.

2. **Key evidence**:
   - DOC_ID: mkt_supply_chain_risk - "Battery supply chains remain concentrated: China dominates cell manufacturing and LFP cathode supply."
   - DOC_ID: mkt_supply_chain_risk - "Falling pack prices help tender economics but do not remove single-country component dependence risk for Indian OEMs."
   - DOC_ID: c_3ec647c0c96c88dbe985 - "India can't move beyond Aamron and Exide batteries."
   - DOC_ID: c_62f5af4a3b0926899117 - "I think sodium ion batteries can be a bit economical. No dependence on china also."
   - DOC_ID: chart_chart_battery_price_trend - "Lithium-ion battery pack price decline: $1183/kWh 2010 to $139 in 2023."

3. **Interpretation**: Th

## 3 · Strategic Output — Comprehensive 3-Year Corporate EV Strategy

Synthesised from Evidence Pack A (the 8 RAG Q&As above) + Evidence Pack B (Phase 2 image-analytics strategic output, findings F1–F6a/N1–N3), structured on the four board pillars, each phased Y1/Y2/Y3.

In [6]:
img_md = (ROOT.parent / "Phase_2_Image_Analytics" / "outputs" / "reports" / "strategic_output.md").read_text()
strategy = R.synthesize_strategy(results, img_md)
(OUT_S / "3_year_corporate_ev_strategy.md").write_text(strategy)
print(f"Wrote {OUT_S/'3_year_corporate_ev_strategy.md'} ({len(strategy):,} chars)")
print()
print(strategy[:1500])

Wrote /home/u_admin/ev_bus/Phase_3_RAG_and_Strategy/outputs/strategy/3_year_corporate_ev_strategy.md (27,241 chars)

# Comprehensive 3-Year Corporate EV Strategy

## Executive Summary

As the Indian automotive OEM prepares to enter the electric bus market, a comprehensive strategy is essential to navigate the competitive landscape dominated by new-age players. This strategy is anchored in the thesis of Trust-backed GCC Growth, focusing on lifetime tender economics while differentiating through fleet uptime guarantees, repair turnaround, depot spares, transparent battery warranties, and strategic charging and financing partnerships.

The strategy is structured around four key pillars mandated by the board:

1. **Market Positioning**
2. **Infrastructure Expansion Partnerships**
3. **Battery Supply-Chain Risk Mitigations**
4. **Digital Marketing Spend Optimization**

Each pillar outlines specific Year 1, Year 2, and Year 3 milestones, accompanied by actionable insights derived from market

## 4 · Structured outputs

| File | Contents |
|---|---|
| `data/rag_index.npz` | 512-d L2-normalised CLIP vectors (one per doc) |
| `data/rag_meta.jsonl` | per-doc metadata (modality, source, features) |
| `data/market_trends.jsonl` | sourced IEA/BNEF/MoHI market facts |
| `data/market_charts/*.png` | chart layouts rendered from those facts |
| `statista/*.png` | real Statista chart exports (indexed as visual layouts) |
| `outputs/executive_queries/executive_queries.{md,json}` | Task 3.2 — 8 queries, retrievals + grounded answers |
| `outputs/strategy/3_year_corporate_ev_strategy.md` | the board-facing Strategic Output |

**Rebuild cycle:** drop new assets (images via Phase 2 rebuild, Statista PNGs into `statista/`, text CSV update) → re-run this notebook.

**Known gaps:** video assets deliberately excluded from this run; Statista chart numbers are indexed via their visual layout + filename caption (no OCR of chart datapoints).